# ADMET Hamiltonian Curation Notebook

This notebook handles ADMET dataset loading, molecular fingerprint generation, feature selection, feature-interaction analysis, and construction of Hamiltonian payloads for downstream LCU/block-encoding workflows.

The notebook stops at Hamiltonian curation and export. QBE generation, LCU validation, block encoding, reflection, and walk-operator logic are handled separately in the class-based LCU framework.

## Reproducible submission configuration
Install `../requirements.txt` before execution. This notebook recomputes all ten TDC datasets, Morgan fingerprints, MI/CMI/triadic statistics, and Stage-6 Hamiltonians without cached interactive state.


Initialization 

In [ ]:
import pandas as pd
import numpy as np

from rdkit import Chem
from rdkit.Chem import AllChem

from tdc.single_pred import ADME

print("Environment ready.")

ADMET Dataset Download 

In [ ]:
from tdc.single_pred import ADME, Tox

admet_datasets = [
    ("ADME", "CYP3A4_Substrate_CarbonMangels"),
    ("ADME", "CYP2D6_Substrate_CarbonMangels"),
    ("ADME", "CYP2C9_Substrate_CarbonMangels"),
    ("Tox", "hERG"),
    ("Tox", "AMES"),
    ("ADME", "BBB_Martins"),
    ("ADME", "PGP_Broccatelli"),
    ("ADME", "Bioavailability_Ma"),
    ("Tox", "DILI"),
    ("ADME", "HIA_Hou"),
]

datasets = {}

for task_type, name in admet_datasets:
    print(f"Downloading {name}...")
    
    if task_type == "ADME":
        data = ADME(name=name)
    elif task_type == "Tox":
        data = Tox(name=name)
    
    df = data.get_data()
    datasets[name] = df
    
    print(name, df.shape, df.columns.tolist())
    display(df.head())

Molecular Fingerprint Generation

In [ ]:
# Morgan count fingerprints for all 10 datasets from PolarisQB
# Maurice-compatible: count-valued features, not binary bit vectors

from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np
import pandas as pd


def smiles_to_morgan_count_fp(smiles, n_bits=1024, radius=2):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    fp = AllChem.GetHashedMorganFingerprint(
        mol,
        radius,
        nBits=n_bits
    )

    arr = np.zeros((n_bits,), dtype=float)

    for idx, count in fp.GetNonzeroElements().items():
        arr[idx] = float(count)

    return arr


fingerprint_data = {}

for name, df in datasets.items():
    print(f"Processing {name}...")

    smiles_col = "Drug"
    label_col = "Y"

    fps = []
    labels = []
    valid_smiles = []

    for _, row in df.iterrows():
        fp = smiles_to_morgan_count_fp(row[smiles_col])

        if fp is not None:
            fps.append(fp)
            labels.append(row[label_col])
            valid_smiles.append(row[smiles_col])

    X = np.array(fps, dtype=float)
    y = np.array(labels)

    fingerprint_data[name] = {
        "X": X,
        "y": y,
        "smiles": valid_smiles,
        "fingerprint_type": "Morgan count fingerprint",
        "n_bits": 1024,
        "radius": 2,
    }

    print(f"{name}: X shape = {X.shape}, y shape = {y.shape}")
    print(f"{name}: min={X.min()}, max={X.max()}, mean={X.mean():.6f}")
    print("-" * 60)

Feature Selection

In [ ]:
# Stage 1: Feature Selection using Mutual Information
from sklearn.feature_selection import mutual_info_classif
import numpy as np
import pandas as pd

top_k = 100

selected_feature_data = {}
mi_summary_rows = []

for name, data in fingerprint_data.items():
    print(f"Processing MI feature selection for {name}...")
    
    X = data["X"].astype(float)
    y = data["y"]
    
    mi_scores = mutual_info_classif(
        X,
        y,
        discrete_features=True,
        random_state=42
    )
    
    feature_ranking = np.argsort(mi_scores)[::-1]
    selected_features = feature_ranking[:top_k]
    
    X_selected = X[:, selected_features]
    
    # MinMax scale selected count features to [0, 1]
    X_min = X_selected.min(axis=0)
    X_max = X_selected.max(axis=0)
    denom = np.where((X_max - X_min) == 0, 1.0, X_max - X_min)
    X_selected_scaled = (X_selected - X_min) / denom
    
    selected_feature_data[name] = {
        "X_selected": X_selected,
        "X_selected_scaled": X_selected_scaled,
        "y": y,
        "selected_features": selected_features,
        "mi_scores": mi_scores,
        "selected_mi_scores": mi_scores[selected_features],
        "smiles": data["smiles"],
        "fingerprint_type": data.get("fingerprint_type", "Morgan count fingerprint"),
        "scaling": {
            "method": "minmax",
            "feature_min": X_min,
            "feature_max": X_max,
        },
    }
    
    mi_summary_rows.append({
        "dataset": name,
        "n_samples": X.shape[0],
        "n_original_features": X.shape[1],
        "n_selected_features": X_selected.shape[1],
        "top_feature": int(selected_features[0]),
        "top_mi_score": float(mi_scores[selected_features[0]]),
        "selected_scaled_min": float(X_selected_scaled.min()),
        "selected_scaled_max": float(X_selected_scaled.max()),
    })
    
    print(f"{name}: selected X shape = {X_selected.shape}")
    print(f"{name}: scaled X shape   = {X_selected_scaled.shape}")
    print(f"{name}: scaled min={X_selected_scaled.min():.6f}, max={X_selected_scaled.max():.6f}")
    print(f"Top feature: {selected_features[0]}, MI = {mi_scores[selected_features[0]]:.6f}")
    print("-" * 60)

mi_summary = pd.DataFrame(mi_summary_rows)
mi_summary

Feature-Feature Interaction

In [ ]:
# Stage 2
# Pairwise Mutual Information Discovery
#
# Computes the pairwise mutual information matrix between the
# Top-100 selected molecular features.
#
# This stage estimates:
#
#     I(X_i ; X_j)
#
# for every unordered feature pair using the NPEET
# k-nearest-neighbour mutual information estimator.
#
# The resulting matrix forms the starting point for:
#
#     Stage 3:
#         Conditional Mutual Information pruning
#
#     Stage 4:
#         Triadic interaction discovery
#
# Notes
# -----
# - Feature ordering is preserved from Stage 1.
# - The interaction matrix is symmetric.
# - Mutual information is theoretically non-negative.
#   Small negative estimates may arise from finite-sample
#   estimation error and are clipped to zero.
# - The output structure remains identical so downstream
#   stages require no structural modification.

import numpy as np
import pandas as pd

from npeet_plus import mi

# Number of nearest neighbours used by the NPEET
# k-nearest-neighbour mutual information estimator.
#
# k = 3 follows the standard KSG recommendation and is
# also the default used by the NPEET implementation.
n_neighbors = 3

pairwise_mi_data = {}

for dataset_name, dataset_data in selected_feature_data.items():

    print(
        f"Computing pairwise mutual information "
        f"for {dataset_name}..."
    )

    # MinMax-scaled Top-100 molecular features.
    X_selected_scaled = np.asarray(
        dataset_data["X_selected_scaled"],
        dtype=float,
    )

    n_features = X_selected_scaled.shape[1]

    # Symmetric pairwise mutual information matrix.
    mi_matrix = np.zeros(
        (n_features, n_features),
        dtype=float,
    )

    # Compute each unordered feature pair only once.
    for feature_i in range(n_features):

        X_i = X_selected_scaled[
            :,
            feature_i,
        ].reshape(-1, 1)

        for feature_j in range(feature_i + 1, n_features):

            X_j = X_selected_scaled[
                :,
                feature_j,
            ].reshape(-1, 1)

            # Estimate pairwise mutual information using
            # the NPEET k-nearest-neighbour estimator.
            #
            # Mutual information is theoretically
            # non-negative. Any small negative estimates
            # arise from finite-sample estimation error and
            # are clipped to zero.
            pairwise_mi = max(
                0.0,
                float(
                    mi(
                        X_i,
                        X_j,
                        k=n_neighbors,
                    )
                ),
            )

            mi_matrix[
                feature_i,
                feature_j,
            ] = pairwise_mi

            mi_matrix[
                feature_j,
                feature_i,
            ] = pairwise_mi

    # Self-information is not used.
    np.fill_diagonal(
        mi_matrix,
        0.0,
    )

    pairwise_mi_data[dataset_name] = {

        "interaction_matrix":
            mi_matrix,

        "mi_matrix":
            mi_matrix,

        "selected_features":
            dataset_data["selected_features"],

        "X_selected":
            dataset_data["X_selected"],

        "X_selected_scaled":
            dataset_data["X_selected_scaled"],

        "y":
            dataset_data["y"],

        "smiles":
            dataset_data["smiles"],
    }

    print(
        f"{dataset_name}: "
        f"MI interaction matrix shape = "
        f"{mi_matrix.shape}"
    )

Check

In [ ]:
print("Symmetric:", np.allclose(mi_matrix, mi_matrix.T))
print("Diagonal:", np.allclose(np.diag(mi_matrix), 0))
print("Minimum:", mi_matrix.min())

In [ ]:
name = "BBB_Martins"

interaction_matrix = pairwise_mi_data[name]["interaction_matrix"]

print("Maximum pairwise MI:", interaction_matrix.max())
print("Mean pairwise MI:", interaction_matrix.mean())
print("Non-zero MI couplings:", np.count_nonzero(interaction_matrix))

In [ ]:
# ============================================================================
# Stage 3
#
# Conditional Mutual Information Pair Selection
#
# For every pair of Top-100 selected molecular features, estimate
#
#     I(X_i ; X_j | Y)
#
# using the nonparametric KSG conditional mutual information estimator
# implemented in npeet_plus.
#
# The estimator measures the statistical dependence between two molecular
# fingerprint features while conditioning on the ADMET class label Y.
#
# Small negative estimates caused by finite-sample estimation error are
# clipped to zero because conditional mutual information is theoretically
# non-negative.
#
# Only feature pairs satisfying
#
#     I(X_i ; X_j | Y) > 0.10
#
# are retained for the sparse interaction Hamiltonian and the subsequent
# triadic-interaction discovery stage.
#
# Important:
#
# - feature_i and feature_j are positions within the Top-100 selected
#   feature matrix.
#
# - original_feature_i and original_feature_j are the corresponding
#   original Morgan fingerprint indices.
#
# - This stage identifies statistically significant interaction supports.
#   Hamiltonian A is constructed later.
# ============================================================================

import numpy as np
import pandas as pd

from npeet_plus import cmi

conditional_mi_threshold = 0.10

conditional_pair_data = {}

conditional_pair_summary_rows = []

for dataset_name, dataset_data in selected_feature_data.items():

    print(
        f"Computing conditional mutual information pairs "
        f"for {dataset_name}..."
    )

    # MinMax-scaled Top-100 molecular features.

    X_selected_scaled = np.asarray(
        dataset_data["X_selected_scaled"],
        dtype=float,
    )

    # Binary ADMET class labels.

    y = np.asarray(
        dataset_data["y"],
        dtype=float,
    ).reshape(-1, 1)

    selected_features = np.asarray(
        dataset_data["selected_features"],
        dtype=int,
    )

    n_samples, n_features = X_selected_scaled.shape

    # Symmetric conditional mutual information matrix.

    conditional_mi_matrix = np.zeros(
        (n_features, n_features),
        dtype=float,
    )

    retained_pairs = []

    # Compute each unordered feature pair exactly once.

    for feature_i in range(n_features):

        X_i = X_selected_scaled[
            :,
            feature_i,
        ].reshape(-1, 1)

        for feature_j in range(feature_i + 1, n_features):

            X_j = X_selected_scaled[
                :,
                feature_j,
            ].reshape(-1, 1)

            # Constant variables contain no conditional mutual information.

            if (
                np.unique(X_i).size < 2
                or np.unique(X_j).size < 2
            ):

                conditional_mi = 0.0

            else:

                conditional_mi = max(
                    0.0,
                    float(
                        cmi(
                            X_i,
                            X_j,
                            y,
                            k=3,
                        )
                    ),
                )

            conditional_mi_matrix[
                feature_i,
                feature_j,
            ] = conditional_mi

            conditional_mi_matrix[
                feature_j,
                feature_i,
            ] = conditional_mi

            # Retain only statistically significant conditional
            # dependencies.

            if conditional_mi > conditional_mi_threshold:

                retained_pairs.append({

                    "feature_i":
                        int(feature_i),

                    "feature_j":
                        int(feature_j),

                    # Original Morgan fingerprint indices.

                    "original_feature_i":
                        int(selected_features[feature_i]),

                    "original_feature_j":
                        int(selected_features[feature_j]),

                    "conditional_mi":
                        float(conditional_mi),

                })

    np.fill_diagonal(
        conditional_mi_matrix,
        0.0,
    )

    # Rank retained interactions from strongest to weakest.

    retained_pairs.sort(
        key=lambda pair: pair["conditional_mi"],
        reverse=True,
    )

    # Features participating in at least one retained interaction.

    active_feature_positions = sorted({

        feature_position

        for pair in retained_pairs

        for feature_position in (

            pair["feature_i"],
            pair["feature_j"],

        )

    })

    active_original_features = [

        int(selected_features[position])

        for position in active_feature_positions

    ]

    conditional_pair_data[dataset_name] = {

        "conditional_mi_threshold":
            conditional_mi_threshold,

        "conditional_mi_matrix":
            conditional_mi_matrix,

        "retained_pairs":
            retained_pairs,

        "num_retained_pairs":
            len(retained_pairs),

        "active_feature_positions":
            active_feature_positions,

        "active_original_features":
            active_original_features,

        "num_active_features":
            len(active_feature_positions),

        # Preserve feature-selection data for Stage 4.

        "X_selected":
            dataset_data["X_selected"],

        "X_selected_scaled":
            X_selected_scaled,

        "y":
            y,

        "selected_features":
            selected_features,

        "smiles":
            dataset_data["smiles"],

    }

    retained_scores = [

        pair["conditional_mi"]

        for pair in retained_pairs

    ]

    conditional_pair_summary_rows.append({

        "dataset":
            dataset_name,

        "n_candidate_pairs":
            n_features * (n_features - 1) // 2,

        "conditional_mi_threshold":
            conditional_mi_threshold,

        "n_retained_pairs":
            len(retained_pairs),

        "n_active_features":
            len(active_feature_positions),

        "strongest_conditional_mi":
            max(retained_scores)
            if retained_scores else 0.0,

        "weakest_retained_conditional_mi":
            min(retained_scores)
            if retained_scores else 0.0,

    })

    print(

        f"{dataset_name}: "

        f"{len(retained_pairs)} retained pairs, "

        f"{len(active_feature_positions)} active features"

    )

    if retained_pairs:

        strongest_pair = retained_pairs[0]

        print(

            "Strongest pair: "

            f"({strongest_pair['feature_i']}, "

            f"{strongest_pair['feature_j']}), "

            f"conditional MI = "

            f"{strongest_pair['conditional_mi']:.6f}"

        )

    else:

        print(

            "No pairs exceeded the conditional MI threshold."

        )

    print("-" * 60)

conditional_pair_summary = pd.DataFrame(
    conditional_pair_summary_rows
)

conditional_pair_summary

In [ ]:
# ============================================================================
# Stage 4
#
# Triadic Interaction Discovery
#
# Stage 3 retained feature pairs whose conditional mutual information
# exceeded the pair-selection threshold:
#
#     I(X_i ; X_j | Y) > 0.10
#
# Stage 4 extends those retained pairs with a third active feature X_k
# and estimates the three-variable interaction information:
#
#     I(X_i ; X_j ; X_k)
#         = I(X_i ; X_j) - I(X_i ; X_j | X_k)
#
# Interaction information is theoretically symmetric, so the score is
# estimated in all three equivalent orientations:
#
#     I(X_i ; X_j) - I(X_i ; X_j | X_k)
#     I(X_i ; X_k) - I(X_i ; X_k | X_j)
#     I(X_j ; X_k) - I(X_j ; X_k | X_i)
#
# The three estimates are averaged to reduce finite-sample and
# nearest-neighbour estimator noise.
#
# Only candidate triads satisfying
#
#     I(X_i ; X_j ; X_k) > 0.15
#
# are retained for subsequent three-body Hamiltonian construction.
#
# Candidate generation is sparse:
#
# - Begin with each retained Stage 3 pair.
# - Extend it using another active Stage 3 feature.
# - Sort and deduplicate each candidate triad.
#
# Important:
#
# - feature_i, feature_j, and feature_k refer to positions within the
#   Top-100 selected feature matrix.
#
# - original_feature_i, original_feature_j, and original_feature_k are
#   the corresponding original Morgan fingerprint indices.
#
# - Positive interaction information indicates shared or redundant
#   three-feature dependence.
#
# - Negative interaction information indicates synergistic dependence.
#   The paper specifies retaining scores exceeding a positive threshold,
#   so negative values are preserved for diagnostics but are not retained.
#
# - This stage identifies three-feature interaction supports only.
#   Three-body Hamiltonian terms are constructed later.
# ============================================================================

import numpy as np
import pandas as pd

from npeet_plus import mi, cmi

triad_threshold = 0.15

triadic_interaction_data = {}

triadic_summary_rows = []

for dataset_name, dataset_data in conditional_pair_data.items():

    print(
        f"Computing triadic interaction information "
        f"for {dataset_name}..."
    )

    # MinMax-scaled Top-100 molecular features.

    X_selected_scaled = np.asarray(
        dataset_data["X_selected_scaled"],
        dtype=float,
    )

    selected_features = np.asarray(
        dataset_data["selected_features"],
        dtype=int,
    )

    retained_pairs = dataset_data["retained_pairs"]

    active_feature_positions = list(
        dataset_data["active_feature_positions"]
    )

    n_samples, n_features = X_selected_scaled.shape

    # ------------------------------------------------------------------------
    # Generate sparse candidate triads.
    #
    # Each retained Stage 3 pair is extended by one additional active
    # feature. Sorting and storing the triads in a set ensures that each
    # unordered triad is evaluated exactly once.
    # ------------------------------------------------------------------------

    candidate_triads = set()

    for pair in retained_pairs:

        feature_i = int(pair["feature_i"])
        feature_j = int(pair["feature_j"])

        for feature_k in active_feature_positions:

            feature_k = int(feature_k)

            if feature_k in (feature_i, feature_j):
                continue

            candidate_triad = tuple(
                sorted(
                    (
                        feature_i,
                        feature_j,
                        feature_k,
                    )
                )
            )

            candidate_triads.add(candidate_triad)

    candidate_triads = sorted(candidate_triads)

    retained_triads = []

    all_triad_scores = []

    # ------------------------------------------------------------------------
    # Estimate interaction information for each sparse candidate triad.
    # ------------------------------------------------------------------------

    for feature_i, feature_j, feature_k in candidate_triads:

        X_i = X_selected_scaled[
            :,
            feature_i,
        ].reshape(-1, 1)

        X_j = X_selected_scaled[
            :,
            feature_j,
        ].reshape(-1, 1)

        X_k = X_selected_scaled[
            :,
            feature_k,
        ].reshape(-1, 1)

        # A constant variable cannot support a meaningful
        # nearest-neighbour interaction-information estimate.

        if (
            np.unique(X_i).size < 2
            or np.unique(X_j).size < 2
            or np.unique(X_k).size < 2
        ):

            interaction_ij_given_k = 0.0
            interaction_ik_given_j = 0.0
            interaction_jk_given_i = 0.0
            triadic_interaction = 0.0

        else:

            # Orientation 1:
            #
            # I(X_i ; X_j ; X_k)
            #     = I(X_i ; X_j) - I(X_i ; X_j | X_k)

            interaction_ij_given_k = float(
                mi(
                    X_i,
                    X_j,
                    k=3,
                )
                -
                cmi(
                    X_i,
                    X_j,
                    X_k,
                    k=3,
                )
            )

            # Orientation 2:
            #
            # I(X_i ; X_k ; X_j)
            #     = I(X_i ; X_k) - I(X_i ; X_k | X_j)

            interaction_ik_given_j = float(
                mi(
                    X_i,
                    X_k,
                    k=3,
                )
                -
                cmi(
                    X_i,
                    X_k,
                    X_j,
                    k=3,
                )
            )

            # Orientation 3:
            #
            # I(X_j ; X_k ; X_i)
            #     = I(X_j ; X_k) - I(X_j ; X_k | X_i)

            interaction_jk_given_i = float(
                mi(
                    X_j,
                    X_k,
                    k=3,
                )
                -
                cmi(
                    X_j,
                    X_k,
                    X_i,
                    k=3,
                )
            )

            # The three expressions are mathematically equivalent.
            # Their average reduces finite-sample estimator variation.

            triadic_interaction = float(
                np.mean(
                    [
                        interaction_ij_given_k,
                        interaction_ik_given_j,
                        interaction_jk_given_i,
                    ]
                )
            )

        all_triad_scores.append(triadic_interaction)

        # Retain only positive interaction information exceeding
        # the documented triad threshold.

        if triadic_interaction > triad_threshold:

            retained_triads.append({

                "feature_i":
                    int(feature_i),

                "feature_j":
                    int(feature_j),

                "feature_k":
                    int(feature_k),

                # Original Morgan fingerprint indices before
                # Top-100 feature reindexing.

                "original_feature_i":
                    int(selected_features[feature_i]),

                "original_feature_j":
                    int(selected_features[feature_j]),

                "original_feature_k":
                    int(selected_features[feature_k]),

                "triadic_interaction":
                    float(triadic_interaction),

                # Preserve the three estimator orientations for
                # methodological validation and debugging.

                "orientation_ij_given_k":
                    float(interaction_ij_given_k),

                "orientation_ik_given_j":
                    float(interaction_ik_given_j),

                "orientation_jk_given_i":
                    float(interaction_jk_given_i),

            })

    # Rank retained triads from strongest to weakest.

    retained_triads.sort(
        key=lambda triad: triad["triadic_interaction"],
        reverse=True,
    )

    # Features participating in at least one retained triad.

    triad_active_feature_positions = sorted({

        feature_position

        for triad in retained_triads

        for feature_position in (

            triad["feature_i"],
            triad["feature_j"],
            triad["feature_k"],

        )

    })

    triad_active_original_features = [

        int(selected_features[position])

        for position in triad_active_feature_positions

    ]

    retained_triad_scores = [

        triad["triadic_interaction"]

        for triad in retained_triads

    ]

    triadic_interaction_data[dataset_name] = {

        "triad_threshold":
            triad_threshold,

        "candidate_triads":
            candidate_triads,

        "num_candidate_triads":
            len(candidate_triads),

        "retained_triads":
            retained_triads,

        "num_retained_triads":
            len(retained_triads),

        "triad_active_feature_positions":
            triad_active_feature_positions,

        "triad_active_original_features":
            triad_active_original_features,

        "num_triad_active_features":
            len(triad_active_feature_positions),

        # Preserve Stage 3 pair-selection outputs.

        "retained_pairs":
            retained_pairs,

        "num_retained_pairs":
            len(retained_pairs),

        "active_feature_positions":
            active_feature_positions,

        "active_original_features":
            dataset_data["active_original_features"],

        "conditional_mi_matrix":
            dataset_data["conditional_mi_matrix"],

        # Preserve molecular and feature-selection data for
        # Hamiltonian construction.

        "X_selected":
            dataset_data["X_selected"],

        "X_selected_scaled":
            X_selected_scaled,

        "y":
            np.asarray(dataset_data["y"]).ravel(),

        "selected_features":
            selected_features,

        "smiles":
            dataset_data["smiles"],

    }

    triadic_summary_rows.append({

        "dataset":
            dataset_name,

        "n_retained_stage3_pairs":
            len(retained_pairs),

        "n_candidate_triads":
            len(candidate_triads),

        "triad_threshold":
            triad_threshold,

        "n_retained_triads":
            len(retained_triads),

        "n_triad_active_features":
            len(triad_active_feature_positions),

        "strongest_triad_interaction":
            max(retained_triad_scores)
            if retained_triad_scores else 0.0,

        "weakest_retained_triad_interaction":
            min(retained_triad_scores)
            if retained_triad_scores else 0.0,

        "most_negative_candidate_interaction":
            min(all_triad_scores)
            if all_triad_scores else 0.0,

    })

    print(
        f"{dataset_name}: "
        f"{len(candidate_triads)} candidate triads, "
        f"{len(retained_triads)} retained triads, "
        f"{len(triad_active_feature_positions)} active triad features"
    )

    if retained_triads:

        strongest_triad = retained_triads[0]

        print(
            "Strongest triad: "
            f"({strongest_triad['feature_i']}, "
            f"{strongest_triad['feature_j']}, "
            f"{strongest_triad['feature_k']}), "
            f"interaction information = "
            f"{strongest_triad['triadic_interaction']:.6f}"
        )

    else:

        print(
            "No triads exceeded the triadic-interaction threshold."
        )

    print("-" * 60)

triadic_summary = pd.DataFrame(
    triadic_summary_rows
)

triadic_summary

In [ ]:
# =========================================================
# Stage 5
# Final Interaction Graph Assembly
#
# Assemble the final sparse molecular interaction graph
# from the retained second-order (Stage 3) and third-order
# (Stage 4) interactions.
#
# This stage performs:
#
#   • Structural validation
#   • Duplicate removal
#   • Compact feature re-indexing
#   • Mapping back to original Morgan fingerprints
#   • Final interaction graph construction
#
# No additional mutual information estimation or
# statistical pruning is performed.
#
# The resulting interaction graph forms the classical
# problem Hamiltonian used for quantum encoding.
# =========================================================

# ---------------------------------------------------------
# Goal
#
# At this point all statistically informative interactions
# have already been identified.
#
# Stage 5 converts these retained interactions into a
# consistent graph representation suitable for Hamiltonian
# construction.
#
# The output consists of:
#
#     • Active molecular features
#     • Pairwise interaction edges
#     • Triadic interaction hyperedges
#     • Compact qubit indexing
#     • Original Morgan feature mapping
#
# These structures are subsequently converted into the
# one-, two- and three-body Pauli Hamiltonian.
# ---------------------------------------------------------

In [ ]:
# =========================================================
# Stage 5
# Final Interaction Hypergraph Assembly
#
# Construct the final sparse interaction hypergraph from the
# retained pairwise (Stage 3) and triadic (Stage 4)
# interactions.
#
# This stage performs:
#
#   • Merge active features
#   • Compact Hamiltonian qubit indexing
#   • Preserve mapping to original Morgan bits
#   • Assemble weighted pair interactions
#   • Assemble weighted triad interactions
#   • Structural validation
#
# No additional statistical analysis is performed.
#
# Output:
# interaction_hypergraph
# =========================================================

import pandas as pd

interaction_hypergraph = {}

summary_rows = []

for name in fingerprint_data.keys():

    print(f"Assembling interaction hypergraph for {name}...")

    # =====================================================
    # Load Stage 3 outputs
    # =====================================================

    pair_data = conditional_pair_data[name]

    retained_pairs = pair_data["retained_pairs"]

    selected_features = pair_data["selected_features"]

    # =====================================================
    # Load Stage 4 outputs
    # =====================================================

    triad_data = triadic_interaction_data[name]

    retained_triads = triad_data["retained_triads"]

    # =====================================================
    # Collect active features
    # =====================================================

    active_features = set()

    for pair in retained_pairs:

        active_features.add(pair["feature_i"])
        active_features.add(pair["feature_j"])

    for triad in retained_triads:

        active_features.add(triad["feature_i"])
        active_features.add(triad["feature_j"])
        active_features.add(triad["feature_k"])

    active_features = sorted(active_features)

    # =====================================================
    # Compact Hamiltonian indexing
    # =====================================================

    compact_index_map = {
        feature: idx
        for idx, feature in enumerate(active_features)
    }

    inverse_index_map = {
        idx: feature
        for feature, idx in compact_index_map.items()
    }

    # =====================================================
    # Original Morgan fingerprint mapping
    # =====================================================

    original_feature_map = {

        compact_index_map[f]: int(selected_features[f])

        for f in active_features

    }

    # =====================================================
    # Pair interactions
    # =====================================================

    pair_terms = []

    for pair in retained_pairs:

        pair_terms.append({

            "qubit_i": compact_index_map[pair["feature_i"]],

            "qubit_j": compact_index_map[pair["feature_j"]],

            "weight": float(pair["conditional_mi"])

        })

    # =====================================================
    # Triad interactions
    # =====================================================

    triad_terms = []

    for triad in retained_triads:

        triad_terms.append({

            "qubit_i": compact_index_map[triad["feature_i"]],

            "qubit_j": compact_index_map[triad["feature_j"]],

            "qubit_k": compact_index_map[triad["feature_k"]],

            "weight": float(triad["triadic_interaction"])

        })

    # =====================================================
    # Validation
    # =====================================================

    assert len(active_features) == len(compact_index_map)

    assert len(compact_index_map) == len(inverse_index_map)

    # =====================================================
    # Store hypergraph
    # =====================================================

    interaction_hypergraph[name] = {

        "active_features": active_features,

        "compact_index_map": compact_index_map,

        "inverse_index_map": inverse_index_map,

        "original_feature_map": original_feature_map,

        "pair_terms": pair_terms,

        "triad_terms": triad_terms

    }

    summary_rows.append({

        "dataset": name,

        "hamiltonian_qubits": len(active_features),

        "pair_terms": len(pair_terms),

        "triad_terms": len(triad_terms)

    })

    print(
        f"Qubits: {len(active_features)} | "
        f"Pairs: {len(pair_terms)} | "
        f"Triads: {len(triad_terms)}"
    )

    print("-" * 60)

stage5_summary = pd.DataFrame(summary_rows)

display(stage5_summary)

# Problem Hamiltonian Construction


The molecular problem Hamiltonian is constructed as the sum of two complementary components:

### Hamiltonian A — Global Feature Interaction Hamiltonian

H_A=\sum_{i<j}MI(X_i,X_j)\,Z_iZ_j

* Constructed once from the training dataset.
* Encodes pairwise statistical dependencies between molecular features using pairwise mutual information.
* Represents the global interaction structure shared across the dataset.


### Hamiltonian B — Molecular Local-Field Hamiltonian

H_B(x)=\sum_i x_iZ_i

* Constructed individually for each molecule.
* Encodes the molecular fingerprint as local Pauli-Z fields.
* Represents the molecule-specific contribution to the Hamiltonian.


## Problem Hamiltonian

The complete molecular Hamiltonian is given by

H_{\mathrm{problem}}(x)=H_A+H_B(x).

This Hamiltonian serves as the input to the counterdiabatic Hamiltonian construction in Notebook 2.

In [ ]:
# =========================================================
# Stage 6A
# Global Interaction Hamiltonian (H_A)
#
# Construct the explicit dataset-level interaction
# Hamiltonian from the interaction hypergraph.
#
# This Hamiltonian represents the learned interaction
# landscape shared by all molecules in a dataset.
#
# H_A contains:
#
#     • Pairwise ZZ interaction terms
#     • Three-body ZZZ interaction terms
#
# Each Hamiltonian term is stored in a backend-agnostic
# representation suitable for later emission into
# Triple Alpha.
#
# No molecule-specific information is introduced here.
#
# Output:
#
# global_hamiltonian
#
# =========================================================

import pandas as pd

global_hamiltonian = {}

summary_rows = []

for name, hypergraph in interaction_hypergraph.items():

    print(f"Constructing H_A for {name}...")

    # =====================================================
    # Load Stage 5 outputs
    # =====================================================

    pair_terms = hypergraph["pair_terms"]

    triad_terms = hypergraph["triad_terms"]

    # =====================================================
    # Hamiltonian term container
    # =====================================================

    hamiltonian_terms = []

    # =====================================================
    # Pair interaction terms
    #
    #     J_ij Z_i Z_j
    # =====================================================

    for pair in pair_terms:

        hamiltonian_terms.append({

            "term_type": "pair",

            "locality": 2,

            "pauli": "ZZ",

            "qubits": (

                pair["qubit_i"],

                pair["qubit_j"]

            ),

            "coefficient": float(pair["weight"])

        })

    # =====================================================
    # Three-body interaction terms
    #
    #     K_ijk Z_i Z_j Z_k
    # =====================================================

    for triad in triad_terms:

        hamiltonian_terms.append({

            "term_type": "triad",

            "locality": 3,

            "pauli": "ZZZ",

            "qubits": (

                triad["qubit_i"],

                triad["qubit_j"],

                triad["qubit_k"]

            ),

            "coefficient": float(triad["weight"])

        })

    # =====================================================
    # Store Hamiltonian
    # =====================================================

    global_hamiltonian[name] = {

        "num_qubits": len(hypergraph["active_features"]),

        "active_features": hypergraph["active_features"],

        "compact_index_map": hypergraph["compact_index_map"],

        "inverse_index_map": hypergraph["inverse_index_map"],

        "original_feature_map": hypergraph["original_feature_map"],

        "terms": hamiltonian_terms

    }

    # =====================================================
    # Summary
    # =====================================================

    summary_rows.append({

        "dataset": name,

        "qubits": len(hypergraph["active_features"]),

        "pair_terms": len(pair_terms),

        "triad_terms": len(triad_terms),

        "total_terms": len(hamiltonian_terms)

    })

    print(

        f"Qubits: {len(hypergraph['active_features'])} | "

        f"Pair Terms: {len(pair_terms)} | "

        f"Triad Terms: {len(triad_terms)} | "

        f"Total Terms: {len(hamiltonian_terms)}"

    )

    print("-" * 60)

# =========================================================
# Stage Summary
# =========================================================

stage6A_summary = pd.DataFrame(summary_rows)

display(stage6A_summary)

In [ ]:
# =========================================================
# Stage 6B
# Parameterized Local Hamiltonian (H_B)
#
# Construct the parameterized one-body Hamiltonian
#
#           H_B(x) = Σ x_i Z_i
#
# where
#
#     x = [x0, x1, ..., xN-1]
#
# is a one-dimensional molecular feature vector.
#
# The Hamiltonian structure is fixed.
# Only the parameter vector changes from molecule to
# molecule.
#
# This produces one reusable Hamiltonian family for each
# dataset.
#
# Output:
#
# parameterized_hamiltonian
#
# =========================================================

import pandas as pd

parameterized_hamiltonian = {}

summary_rows = []

for name, hypergraph in interaction_hypergraph.items():

    print(f"Constructing parameterized H_B for {name}...")

    # =====================================================
    # Load retained feature mapping
    # =====================================================

    active_features = hypergraph["active_features"]

    num_qubits = len(active_features)

    # =====================================================
    # Construct parameterized one-body terms
    # =====================================================

    local_terms = []

    for qubit in range(num_qubits):

        local_terms.append({

            "term_type": "local",

            "locality": 1,

            "pauli": "Z",

            "qubits": (qubit,),

            # Coefficient = x[parameter_index]
            "parameter_index": qubit

        })

    # =====================================================
    # Store parameterized Hamiltonian
    # =====================================================

    parameterized_hamiltonian[name] = {

        # Hamiltonian dimensions
        "num_qubits": num_qubits,

        "num_parameters": num_qubits,

        # Parameter vector definition
        "parameter_vector": {

            "name": "x",

            "dimension": num_qubits,

            "values": None

        },

        # Feature mappings
        "active_features": active_features,

        "compact_index_map": hypergraph["compact_index_map"],

        "inverse_index_map": hypergraph["inverse_index_map"],

        "original_feature_map": hypergraph["original_feature_map"],

        # Parameterized Hamiltonian terms
        "terms": local_terms

    }

    summary_rows.append({

        "dataset": name,

        "qubits": num_qubits,

        "parameters": num_qubits,

        "local_terms": len(local_terms)

    })

    print(

        f"Qubits: {num_qubits} | "

        f"Parameters: {num_qubits} | "

        f"Local Terms: {len(local_terms)}"

    )

    print("-" * 60)

# =========================================================
# Stage Summary
# =========================================================

stage6B_summary = pd.DataFrame(summary_rows)

display(stage6B_summary)

In [ ]:
# =========================================================
# Stage 6C
# Parameterized Problem Hamiltonian
#
# Construct the complete parameterized Hamiltonian
#
#           H_problem(x) = H_A + H_B(x)
#
# where
#
#     H_A     : fixed interaction Hamiltonian
#
#     H_B(x)  : parameterized local Hamiltonian
#
# The resulting Hamiltonian is backend independent and
# serves as the input to the later AGP, counterdiabatic,
# qubitization and QSVT stages.
#
# Output:
#
# problem_hamiltonian
#
# =========================================================

import pandas as pd

problem_hamiltonian = {}

summary_rows = []

for name in global_hamiltonian.keys():


    print(f"Constructing H_problem for {name}...")

    HA = global_hamiltonian[name]
    HB = parameterized_hamiltonian[name]

    # =====================================================
    # Consistency checks
    # =====================================================

    assert HA["num_qubits"] == HB["num_qubits"], \
        f"Qubit mismatch for {name}"

    # =====================================================
    # Merge Hamiltonians
    # =====================================================

    problem_terms = HA["terms"] + HB["terms"]

    problem_hamiltonian[name] = {

        # Dimensions
        "num_qubits": HA["num_qubits"],

        "num_parameters": HB["num_parameters"],

        # Parameter vector
        "parameter_vector": HB["parameter_vector"],

        # Feature mappings
        "active_features": HA["active_features"],

        "compact_index_map": HA["compact_index_map"],

        "inverse_index_map": HA["inverse_index_map"],

        "original_feature_map": HA["original_feature_map"],

        # Component Hamiltonians
        "interaction_terms": HA["terms"],

        "local_terms": HB["terms"],

        # Complete Hamiltonian
        "terms": problem_terms

    }

    summary_rows.append({

        "dataset": name,

        "qubits": HA["num_qubits"],

        "interaction_terms": len(HA["terms"]),

        "local_terms": len(HB["terms"]),

        "total_terms": len(problem_terms),

        "parameters": HB["num_parameters"]

    })

    print(

        f"Qubits: {HA['num_qubits']} | "

        f"Interaction Terms: {len(HA['terms'])} | "

        f"Local Terms: {len(HB['terms'])} | "

        f"Total Terms: {len(problem_terms)}"

    )

    print("-" * 60)

# =========================================================
# Stage Summary
# =========================================================

stage6C_summary = pd.DataFrame(summary_rows)

display(stage6C_summary)

# CD HAMILTONIAN 

# Stage 7
# Parameterized AGP Basis Validation

Before constructing the counterdiabatic Hamiltonian, we verify whether the
molecular problem Hamiltonian and its AGP form fixed parameterized operator
families.

For each dataset, the problem Hamiltonian is

\[
H_{\mathrm{problem}}(\theta)
=
\sum_i \theta_i Z_i
+
\sum_{(i,j)\in E} J_{ij} Z_i Z_j
+
\sum_{(i,j,k)\in T} K_{ijk} Z_i Z_j Z_k.
\]

The molecular fingerprint parameters \(\theta_i\) affect only the coefficients
of the one-body \(Z_i\) terms.

The operator support

\[
\mathcal{B}_{\mathrm{problem}}
=
\{Z_i\}
\cup
\{Z_i Z_j\}_{(i,j)\in E}
\cup
\{Z_i Z_j Z_k\}_{(i,j,k)\in T}
\]

is fixed for a dataset because the active qubits, retained interaction pairs,
and retained interaction triads are constructed once from the training data.

Therefore, different molecules instantiate different coefficient vectors over
the same Pauli-operator basis.

In [ ]:
# =========================================================
# Stage 7A
# Fixed Problem-Hamiltonian Operator Family
#
# Verify that H_problem(theta) has:
#
#     • fixed Pauli support,
#     • fixed qubit support,
#     • fixed interaction coefficients,
#     • molecule-dependent local coefficients only.
#
# =========================================================

problem_operator_family = {}

for dataset_name, hamiltonian in problem_hamiltonian.items():

    interaction_basis = []
    local_basis = []

    # Fixed dataset-level interaction basis
    for interaction_index, term in enumerate(hamiltonian["interaction_terms"]):

        interaction_basis.append({

            "interaction_index": interaction_index,

            "pauli": term["pauli"],

            "qubits": tuple(term["qubits"]),

            "locality": int(term["locality"]),

            "coefficient": float(term["coefficient"]),

        })
    # Fixed local operator basis with parameterized coefficients
    for term in hamiltonian["local_terms"]:

        local_basis.append({
            "pauli": term["pauli"],
            "qubits": tuple(term["qubits"]),
            "locality": int(term["locality"]),
            "parameter_index": int(term["parameter_index"]),
        })

    complete_operator_support = tuple(
        [
            (term["pauli"], tuple(term["qubits"]))
            for term in interaction_basis
        ]
        +
        [
            (term["pauli"], tuple(term["qubits"]))
            for term in local_basis
        ]
    )

    # Every molecular parameter must map to exactly one local Z term.
    parameter_indices = sorted(
        term["parameter_index"]
        for term in local_basis
    )

    expected_parameter_indices = list(
        range(hamiltonian["num_parameters"])
    )

    assert parameter_indices == expected_parameter_indices

    # Every local parameterized operator must be a one-body Z term.
    assert all(
        term["pauli"] == "Z"
        and term["locality"] == 1
        and len(term["qubits"]) == 1
        for term in local_basis
    )

    problem_operator_family[dataset_name] = {

        "num_qubits":
            hamiltonian["num_qubits"],

        "num_parameters":
            hamiltonian["num_parameters"],

        "interaction_basis":
            interaction_basis,

        "local_basis":
            local_basis,

        "operator_support":
            complete_operator_support,

        "num_interaction_operators":
            len(interaction_basis),

        "num_local_operators":
            len(local_basis),

        "num_total_operators":
            len(complete_operator_support),

        "fixed_operator_family":
            True,

        "parameter_dependence":
            "Only local Z coefficients depend on theta.",
    }

    print(dataset_name)
    print(
        "Fixed interaction operators:",
        len(interaction_basis)
    )
    print(
        "Parameterized local operators:",
        len(local_basis)
    )
    print(
        "Total fixed operator support:",
        len(complete_operator_support)
    )
    print(
        "Validated fixed operator family:",
        True
    )
    print("-" * 60)

## Stage 7B
### Nested Commutator Operator Basis

The adiabatic gauge potential (AGP) is constructed from the operator basis
generated by repeated commutators of the problem Hamiltonian.

Rather than deriving the AGP directly, we first determine the operator family
generated by

\[
H_{\mathrm{problem}}(\theta)
\]

under successive commutation with an initial driver Hamiltonian.

This follows the validated counterdiabatic construction and determines the
fixed operator basis used in the AGP expansion.

In [ ]:
# ============================================================
# Stage 7B
# Initial Hamiltonian for AGP construction
# ============================================================

initial_hamiltonians = {}

for dataset_name, H in problem_operator_family.items():

    n = H["num_qubits"]

    initial_terms = []

    for q in range(n):
        initial_terms.append({
            "pauli": "X",
            "qubits": (q,),
            "coefficient": 1.0
        })

    initial_hamiltonians[dataset_name] = {
        "num_qubits": n,
        "terms": initial_terms
    }

    print(dataset_name)
    print("Initial Hamiltonian:", len(initial_terms), "X terms")
    print("-"*60)

## Stage 7C — Adiabatic Hamiltonian

Following the validated counterdiabatic methodology, define the adiabatic interpolation

$begin:math:display$
H\_\{\\mathrm\{ad\}\}\(\\lambda\,\\theta\)
\=
\(1\-\\lambda\)H\_\{\\mathrm\{initial\}\}
\+
\\lambda H\_\{\\mathrm\{problem\}\}\(\\theta\)\,
$end:math:display$

where

- $begin:math:text$H\_\{\\mathrm\{initial\}\}$end:math:text$ is the transverse-field driver Hamiltonian,
- $begin:math:text$H\_\{\\mathrm\{problem\}\}\(\\theta\)$end:math:text$ is the parameterized molecular Hamiltonian,
- $begin:math:text$\\lambda\\in\[0\,1\]$end:math:text$ is the interpolation parameter.

The zeroth-order operator is

$begin:math:display$
O\_0
\=
\\frac\{\\partial H\_\{\\mathrm\{ad\}\}\}\{\\partial\\lambda\}
\=
H\_\{\\mathrm\{problem\}\}\(\\theta\)
\-
H\_\{\\mathrm\{initial\}\}\.
$end:math:display$

No commutators are evaluated in this stage. This stage defines the operator objects required for the subsequent AGP construction.

In [ ]:
# ============================================================
# Stage 7C
# Adiabatic Hamiltonian Definition
# Fixed Linear Interpolation
# ============================================================

adiabatic_hamiltonians = {}
O0_operators = {}

for dataset_name in problem_operator_family:

    H_problem = problem_operator_family[dataset_name]
    H_initial = initial_hamiltonians[dataset_name]

    adiabatic_hamiltonians[dataset_name] = {

        # Component Hamiltonians
        "H_initial": H_initial,
        "H_problem": H_problem,

        # Fixed interpolation used for every molecule
        "interpolation": {
            "type": "fixed_linear",
            "formula": "(1-lambda) * H_initial + lambda * H_problem",
            "runtime_parameter": False
        }

    }

    # O0 = dH_ad / dλ = H_problem − H_initial
    O0_operators[dataset_name] = {

        "positive": H_problem,
        "negative": H_initial,

        "derivative_variable": "lambda",
        "runtime_parameter": False

    }

    print(dataset_name)
    print("✓ Fixed linear H_ad defined")
    print("✓ O0 = H_problem − H_initial")
    print("✓ λ fixed by construction (not a runtime input)")
    print("-" * 60)

## Stage 7D — First-Order Adiabatic Gauge Potential Operator

The first-order operator in the nested-commutator expansion is obtained by evaluating the commutator between the initial Hamiltonian and the parameterized problem Hamiltonian,

$begin:math:display$
O\_1\=\[H\_\{\\mathrm\{initial\}\}\,H\_\{\\mathrm\{problem\}\}\]\.
$end:math:display$

Following the validated counterdiabatic implementation, each Pauli term of the initial Hamiltonian is commuted with every operator in the problem Hamiltonian using PennyLane's symbolic Pauli algebra. Terms with vanishing commutators are discarded.

Unlike the original implementation, the problem Hamiltonian is stored as separate interaction and local operator bases. These bases are combined only for the commutator evaluation, preserving the modular operator-family representation introduced in the previous stages.

The resulting non-zero commutators form the first-order operator $begin:math:text$O\_1$end:math:text$, which serves as the input to the second nested commutator in the subsequent stage.

In [ ]:
# ============================================================
# Stage 7D
# BUILD FIRST-ORDER OPERATOR
# O1 = [H_initial, H_problem]
# Symbolic Operator Representation
# ============================================================

import pennylane as qml
from pennylane.pauli import PauliWord

O1 = {}

for dataset_name, H_problem in problem_operator_family.items():

    print(f"Processing {dataset_name}...")

    problem_terms = (
        H_problem["interaction_basis"]
        + H_problem["local_basis"]
    )

    dataset_results = []

    num_qubits = H_problem["num_qubits"]

    for x in range(num_qubits):

        initial = PauliWord({x: "X"})

        for term in problem_terms:

            problem = PauliWord({

                qubit: pauli

                for qubit, pauli in zip(
                    term["qubits"],
                    term["pauli"]
                )

            })

            comm = qml.commutator(
                initial,
                problem,
                pauli=True
            )

            if len(comm) == 0:
                continue

            # --------------------------------------------------
            # Build symbolic metadata for downstream stages
            # --------------------------------------------------

            result = {

                "initial_qubit": x,

                "commutator": comm,

                "pauli": term["pauli"],

                "qubits": term["qubits"],

                "locality": term["locality"]

            }

            # Fixed interaction operator
            if "coefficient" in term:

                result["term_type"] = "interaction"
                result["coefficient"] = term["coefficient"]
                result["interaction_index"] = term["interaction_index"]

            # Parameterized local operator
            elif "parameter_index" in term:

                result["term_type"] = "local"
                result["parameter_index"] = term["parameter_index"]

            else:

                raise ValueError(
                    f"Unknown operator type encountered in {dataset_name}: {term}"
                )

            dataset_results.append(result)

    O1[dataset_name] = dataset_results

    interaction_count = sum(
        r["term_type"] == "interaction"
        for r in dataset_results
    )

    local_count = sum(
        r["term_type"] == "local"
        for r in dataset_results
    )

    print(f"Non-zero commutators: {len(dataset_results)}")
    print(f"  Interaction operators : {interaction_count}")
    print(f"  Local operators       : {local_count}")
    print("-" * 60)

In [ ]:
# ============================================================
# Stage 7D Validation
# Structural Validation of Symbolic O1
# ============================================================

print("=" * 70)
print("VALIDATING SYMBOLIC FIRST-ORDER OPERATOR (O1)")
print("=" * 70)

total_entries = 0
total_interactions = 0
total_locals = 0

for dataset_name, entries in O1.items():

    print(f"\n{dataset_name}")

    interaction_count = 0
    local_count = 0

    for i, entry in enumerate(entries):

        # ----------------------------------------------------
        # Validation 1
        # term_type must exist
        # ----------------------------------------------------
        assert "term_type" in entry, (
            f"{dataset_name}: Entry {i} missing 'term_type'."
        )

        assert entry["term_type"] in ("interaction", "local"), (
            f"{dataset_name}: Invalid term_type '{entry['term_type']}'."
        )

        # ----------------------------------------------------
        # Validation 2
        # Required metadata
        # ----------------------------------------------------
        for field in (
            "initial_qubit",
            "commutator",
            "pauli",
            "qubits",
            "locality"
        ):
            assert field in entry, (
                f"{dataset_name}: Entry {i} missing '{field}'."
            )

        # ----------------------------------------------------
        # Validation 3
        # Interaction operators
        # ----------------------------------------------------
        if entry["term_type"] == "interaction":

            interaction_count += 1

            assert "coefficient" in entry, (
                f"{dataset_name}: Interaction entry {i} missing coefficient."
            )

            assert "parameter_index" not in entry, (
                f"{dataset_name}: Interaction entry {i} unexpectedly has parameter_index."
            )

            assert isinstance(entry["coefficient"], (int, float)), (
                f"{dataset_name}: Interaction coefficient is not numeric."
            )

        # ----------------------------------------------------
        # Validation 4
        # Local operators
        # ----------------------------------------------------
        else:

            local_count += 1

            assert "parameter_index" in entry, (
                f"{dataset_name}: Local entry {i} missing parameter_index."
            )

            assert "coefficient" not in entry, (
                f"{dataset_name}: Local entry {i} unexpectedly has coefficient."
            )

            assert isinstance(entry["parameter_index"], int), (
                f"{dataset_name}: parameter_index is not an integer."
            )

        # ----------------------------------------------------
        # Validation 5
        # Pauli consistency
        # ----------------------------------------------------
        assert len(entry["pauli"]) == len(entry["qubits"]), (
            f"{dataset_name}: Pauli/qubit length mismatch."
        )

        assert entry["locality"] == len(entry["qubits"]), (
            f"{dataset_name}: Locality mismatch."
        )

        # ----------------------------------------------------
        # Validation 6
        # Non-empty commutator
        # ----------------------------------------------------
        assert len(entry["commutator"]) > 0, (
            f"{dataset_name}: Empty commutator detected."
        )

    # --------------------------------------------------------
    # Dataset summary validation
    # --------------------------------------------------------

    assert interaction_count + local_count == len(entries), (
        f"{dataset_name}: Count mismatch."
    )

    print(f"  Total entries        : {len(entries)}")
    print(f"  Interaction entries  : {interaction_count}")
    print(f"  Local entries        : {local_count}")
    print("   Passed")

    total_entries += len(entries)
    total_interactions += interaction_count
    total_locals += local_count

# ============================================================
# Global validation
# ============================================================

assert total_entries == total_interactions + total_locals

print("\n" + "=" * 70)
print("GLOBAL VALIDATION PASSED")
print("=" * 70)

print(f"Datasets validated     : {len(O1)}")
print(f"Total entries          : {total_entries}")
print(f"Interaction entries    : {total_interactions}")
print(f"Local entries          : {total_locals}")

print("\n All O1 entries contain complete symbolic metadata.")
print(" Interaction and local operators are mutually exclusive.")
print(" Pauli metadata is structurally consistent.")
print(" Symbolic O1 construction validated successfully.")

## Stage 7E — Canonical First-Order Operator

The raw commutators generated in the previous stage are converted into a canonical operator representation.

Each non-zero Pauli commutator is expanded into individual Pauli terms, multiplied by the corresponding problem-Hamiltonian coefficient, and accumulated into a single operator dictionary. Duplicate Pauli operators are merged by summing their coefficients, while vanishing coefficients are discarded.

The resulting operator is the first-order nested commutator

$begin:math:display$
O\_1\=\[H\_\{\\mathrm\{initial\}\}\,H\_\{\\mathrm\{problem\}\}\]\,
$end:math:display$

stored in the same Hamiltonian representation used throughout the remainder of the counterdiabatic construction.

In [ ]:
# ============================================================
# Stage 7E
# FIRST-ORDER OPERATOR
# O1 = [H_initial, H_problem]
# Symbolic Hamiltonian Representation
# ============================================================

O1_hamiltonians = {}

for dataset_name, commutators in O1.items():

    pauli_terms = []

    for item in commutators:

        for pw, coeff in item["commutator"].items():

            qubits = tuple(sorted(pw.keys()))

            pauli = "".join(
                pw[q]
                for q in qubits
            )

            term = {

                "commutator_coefficient": coeff,

                "pauli": pauli,

                "qubits": qubits,

                "locality": len(qubits)

            }

            # --------------------------------------------------
            # Propagate symbolic metadata
            # --------------------------------------------------

            if item["term_type"] == "interaction":

                term["term_type"] = "interaction"
                term["coefficient"] = item["coefficient"]
                term["interaction_index"] = item["interaction_index"]

            elif item["term_type"] == "local":

                term["term_type"] = "local"
                term["parameter_index"] = item["parameter_index"]

            else:

                raise ValueError(
                    f"Unknown term_type: {item['term_type']}"
                )

            pauli_terms.append(term)

    O1_hamiltonians[dataset_name] = {

        "hamiltonian_type": "O1",

        "formula": "O1 = [H_initial, H_problem]",

        "num_qubits":
            problem_operator_family[dataset_name]["num_qubits"],

        "pauli_terms":
            pauli_terms

    }

    print(
        dataset_name,
        "->",
        len(pauli_terms),
        "symbolic Pauli terms"
    )

In [ ]:
# ============================================================
# Stage 7E Validation
# Rigorous Validation of Symbolic O1 Hamiltonians
# ============================================================

import numbers
import numpy as np

print("=" * 74)
print("VALIDATING SYMBOLIC O1 HAMILTONIANS")
print("Stage 7D O1  --->  Stage 7E O1_hamiltonians")
print("=" * 74)

datasets_checked = 0
source_entries_checked = 0
expanded_terms_checked = 0

global_interactions = 0
global_locals = 0

for dataset_name, source_entries in O1.items():

    print(f"\n{dataset_name}")

    generated_terms = O1_hamiltonians[dataset_name]["pauli_terms"]

    # --------------------------------------------------------
    # Validation 1
    # Expected number of expanded terms
    # --------------------------------------------------------

    expected_terms = sum(
        len(entry["commutator"])
        for entry in source_entries
    )

    assert len(generated_terms) == expected_terms, (
        f"{dataset_name}: Expected {expected_terms} expanded terms "
        f"but found {len(generated_terms)}."
    )

    source_entries_checked += len(source_entries)

    interaction_count = 0
    local_count = 0

    # --------------------------------------------------------
    # Validation 2
    # Every expanded term
    # --------------------------------------------------------

    for idx, term in enumerate(generated_terms):

        # Required fields

        for field in (
            "commutator_coefficient",
            "pauli",
            "qubits",
            "locality",
            "term_type",
        ):
            assert field in term, (
                f"{dataset_name}: Term {idx} missing '{field}'."
            )

        # Numeric commutator coefficient

        assert isinstance(
            term["commutator_coefficient"],
            numbers.Number,
        ), (
            f"{dataset_name}: Non-numeric commutator coefficient."
        )

        # Pauli/qubit consistency

        assert len(term["pauli"]) == len(term["qubits"]), (
            f"{dataset_name}: Pauli/qubit mismatch."
        )

        assert term["locality"] == len(term["qubits"]), (
            f"{dataset_name}: Locality mismatch."
        )

        # Metadata propagation

        if term["term_type"] == "interaction":

            interaction_count += 1

            assert "coefficient" in term
            assert "parameter_index" not in term

            assert isinstance(
                term["coefficient"],
                numbers.Number,
            )

        elif term["term_type"] == "local":

            local_count += 1

            assert "parameter_index" in term
            assert "coefficient" not in term

            assert isinstance(
                term["parameter_index"],
                (int, np.integer),
            )

        else:

            raise AssertionError(
                f"{dataset_name}: Unknown term_type '{term['term_type']}'."
            )

    expanded_terms_checked += len(generated_terms)

    global_interactions += interaction_count
    global_locals += local_count

    print(f"  Source commutators : {len(source_entries)}")
    print(f"  Expanded terms     : {len(generated_terms)}")
    print(f"  Interaction terms  : {interaction_count}")
    print(f"  Local terms        : {local_count}")
    print("  ✓ Passed")

    datasets_checked += 1

# ------------------------------------------------------------
# Global validation
# ------------------------------------------------------------

assert expanded_terms_checked == (
    global_interactions + global_locals
)

print("\n" + "=" * 74)
print("GLOBAL VALIDATION PASSED")
print("=" * 74)

print(f"Datasets validated     : {datasets_checked}")
print(f"Source O1 entries      : {source_entries_checked}")
print(f"Expanded Pauli terms   : {expanded_terms_checked}")
print(f"Interaction terms      : {global_interactions}")
print(f"Local terms            : {global_locals}")

print("\n✓ Every Stage 7D entry expanded correctly.")
print("✓ Every symbolic Pauli term contains complete metadata.")
print("✓ Interaction/local metadata preserved.")
print("✓ Commutator coefficients validated.")
print("✓ Symbolic O1 Hamiltonians validated successfully.")

In [ ]:
# ============================================================
# Stage 7E Validation II
# Exact Mathematical Equivalence:
# Stage 7D PauliSentence  <==>  Stage 7E Symbolic Hamiltonian
# ============================================================

print("=" * 74)
print("VALIDATING EXACT MATHEMATICAL EQUIVALENCE")
print("Stage 7D PauliSentence  <==>  Stage 7E Symbolic Hamiltonian")
print("=" * 74)

datasets_checked = 0
terms_checked = 0

for dataset_name in O1:

    print(f"\n{dataset_name}")

    symbolic_terms = O1_hamiltonians[dataset_name]["pauli_terms"]

    output_index = 0

    for source_entry in O1[dataset_name]:

        for pw, coeff in source_entry["commutator"].items():

            assert output_index < len(symbolic_terms), (
                f"{dataset_name}: Missing symbolic term "
                f"at index {output_index}."
            )

            symbolic = symbolic_terms[output_index]

            # --------------------------------------------------
            # Expected values reconstructed directly
            # from the original PauliSentence
            # --------------------------------------------------

            expected_qubits = tuple(sorted(pw.keys()))

            expected_pauli = "".join(
                pw[q]
                for q in expected_qubits
            )

            expected_locality = len(expected_qubits)

            # --------------------------------------------------
            # Exact Pauli operator
            # --------------------------------------------------

            assert symbolic["qubits"] == expected_qubits, (
                f"{dataset_name}: Qubit mismatch "
                f"at symbolic term {output_index}."
            )

            assert symbolic["pauli"] == expected_pauli, (
                f"{dataset_name}: Pauli string mismatch "
                f"at symbolic term {output_index}."
            )

            assert symbolic["locality"] == expected_locality, (
                f"{dataset_name}: Locality mismatch "
                f"at symbolic term {output_index}."
            )

            # --------------------------------------------------
            # Exact commutator coefficient
            # --------------------------------------------------

            assert (
                symbolic["commutator_coefficient"] == coeff
            ), (
                f"{dataset_name}: Commutator coefficient mismatch "
                f"at symbolic term {output_index}."
            )

            output_index += 1
            terms_checked += 1

    assert output_index == len(symbolic_terms), (
        f"{dataset_name}: Stage 7E contains extra symbolic terms."
    )

    print(
        f"  Verified {output_index} symbolic Pauli operators ✓"
    )

    datasets_checked += 1

print("\n" + "=" * 74)
print("EXACT MATHEMATICAL VALIDATION PASSED")
print("=" * 74)

print(f"Datasets validated : {datasets_checked}")
print(f"Pauli operators    : {terms_checked}")

print("\n✓ Every PauliWord is identical.")
print("✓ Every qubit mapping is identical.")
print("✓ Every locality is identical.")
print("✓ Every commutator coefficient is identical.")
print("✓ Stage 7E is a lossless symbolic representation of Stage 7D.")

# Parametirzation of CD-Hamiltonian

In [ ]:
# ============================================================
# Pre-Stage 7F Validation A
# Runtime Coefficient Completeness
# ============================================================

print("=" * 74)
print("PRE-STAGE 7F VALIDATION A")
print("Runtime Coefficient Completeness")
print("=" * 74)

datasets_checked = 0
terms_checked = 0
interaction_terms = 0
local_terms = 0

for dataset_name, hamiltonian in O1_hamiltonians.items():

    print(f"\n{dataset_name}")

    symbolic_terms = hamiltonian["pauli_terms"]

    for term in symbolic_terms:

        # Every symbolic term must carry a commutator coefficient
        assert "commutator_coefficient" in term, (
            f"{dataset_name}: Missing commutator coefficient."
        )

        if term["term_type"] == "interaction":

            interaction_terms += 1

            assert "coefficient" in term, (
                f"{dataset_name}: Interaction term missing coefficient."
            )

            assert "parameter_index" not in term, (
                f"{dataset_name}: Interaction term should not contain parameter_index."
            )

        elif term["term_type"] == "local":

            local_terms += 1

            assert "parameter_index" in term, (
                f"{dataset_name}: Local term missing parameter_index."
            )

            assert "coefficient" not in term, (
                f"{dataset_name}: Local term should not contain coefficient."
            )

        else:
            raise AssertionError(
                f"{dataset_name}: Unknown term_type {term['term_type']}"
            )

        terms_checked += 1

    print(f"  Runtime-ready symbolic terms: {len(symbolic_terms)} ✓")

    datasets_checked += 1

print("\n" + "=" * 74)
print("PRE-STAGE 7F VALIDATION A PASSED")
print("=" * 74)

print(f"Datasets checked    : {datasets_checked}")
print(f"Terms checked       : {terms_checked}")
print(f"Interaction terms   : {interaction_terms}")
print(f"Local terms         : {local_terms}")

print("\n✓ Every runtime coefficient is computable.")
print("✓ Interaction terms contain interaction coefficients.")
print("✓ Local terms contain feature indices.")
print("✓ Symbolic representation is runtime complete.")

In [ ]:
# ============================================================
# Pre-Stage 7F Validation B
# Fixed Operator Basis Per Dataset
# ============================================================

print("=" * 74)
print("PRE-STAGE 7F VALIDATION B")
print("Fixed Operator Basis Per Dataset")
print("=" * 74)

datasets_checked = 0
operators_checked = 0

dataset_operator_bases = {}

for dataset_name, hamiltonian in O1_hamiltonians.items():

    print(f"\n{dataset_name}")

    symbolic_terms = hamiltonian["pauli_terms"]

    operator_basis = []

    for term in symbolic_terms:

        operator_basis.append(
            (
                term["pauli"],
                term["qubits"],
                term["locality"],
                term["term_type"],
            )
        )

    # Ensure no accidental duplicates
    assert len(operator_basis) == len(symbolic_terms), (
        f"{dataset_name}: Operator basis size mismatch."
    )

    dataset_operator_bases[dataset_name] = tuple(operator_basis)

    operators_checked += len(operator_basis)

    print(f"  Fixed operator basis size: {len(operator_basis)} ✓")

    datasets_checked += 1

print("\n" + "=" * 74)
print("PRE-STAGE 7F VALIDATION B PASSED")
print("=" * 74)

print(f"Datasets checked : {datasets_checked}")
print(f"Operators stored : {operators_checked}")

print("\n✓ Every dataset has a fixed operator basis.")
print("✓ Pauli operators are fixed.")
print("✓ Qubit mappings are fixed.")
print("✓ Localities are fixed.")
print("✓ Term ordering is fixed.")
print("✓ Only coefficients need to change during runtime.")

In [ ]:
# ============================================================
# Pre-Stage 7F Validation C
# Duplicate Symbolic Operator Detection
# ============================================================

from collections import defaultdict

print("=" * 74)
print("PRE-STAGE 7F VALIDATION C")
print("Duplicate Symbolic Operator Detection")
print("=" * 74)

datasets_checked = 0
total_duplicate_groups = 0
total_duplicate_terms = 0

duplicate_summary = {}

for dataset_name, hamiltonian in O1_hamiltonians.items():

    print(f"\n{dataset_name}")

    symbolic_terms = hamiltonian["pauli_terms"]

    groups = defaultdict(list)

    for idx, term in enumerate(symbolic_terms):

        key = (
            tuple(term["pauli"]),
            tuple(term["qubits"])
        )

        groups[key].append(idx)

    duplicate_groups = {
        key: ids
        for key, ids in groups.items()
        if len(ids) > 1
    }

    duplicate_summary[dataset_name] = duplicate_groups

    if duplicate_groups:

        print(f"  Duplicate operator groups: {len(duplicate_groups)}")

        duplicate_terms = sum(len(v) for v in duplicate_groups.values())

        print(f"  Duplicate symbolic terms : {duplicate_terms}")

        total_duplicate_groups += len(duplicate_groups)
        total_duplicate_terms += duplicate_terms

        # show first few duplicates
        for i, (key, ids) in enumerate(duplicate_groups.items()):

            if i == 5:
                print("    ...")
                break

            pauli, qubits = key

            print(f"    {pauli} on {qubits} -> entries {ids}")

    else:

        print("  No duplicate symbolic operators ✓")

    datasets_checked += 1

print("\n" + "=" * 74)
print("PRE-STAGE 7F VALIDATION C COMPLETE")
print("=" * 74)

print(f"Datasets checked        : {datasets_checked}")
print(f"Duplicate groups found  : {total_duplicate_groups}")
print(f"Duplicate symbolic terms: {total_duplicate_terms}")

if total_duplicate_groups == 0:

    print("\n✓ Every symbolic Pauli operator is unique.")
    print("✓ No symbolic merging is possible.")

else:

    print("\n⚠ Duplicate symbolic operators detected.")
    print("⚠ Candidate terms for coefficient merging.")

In [ ]:
# ============================================================
# Pre-Stage 7F Validation D
# Canonical Operator Representation
# ============================================================

print("=" * 74)
print("PRE-STAGE 7F VALIDATION D")
print("Canonical Operator Representation")
print("=" * 74)

datasets_checked = 0
terms_checked = 0

for dataset_name, hamiltonian in O1_hamiltonians.items():

    print(f"\n{dataset_name}")

    symbolic_terms = hamiltonian["pauli_terms"]

    for term in symbolic_terms:

        qubits = term["qubits"]
        pauli = term["pauli"]

        # Same length
        assert len(qubits) == len(pauli), (
            f"{dataset_name}: Pauli/qubit length mismatch."
        )

        # Canonical ordering
        assert tuple(qubits) == tuple(sorted(qubits)), (
            f"{dataset_name}: Non-canonical qubit ordering found.\n"
            f"Qubits = {qubits}\n"
            f"Pauli  = {pauli}"
        )

        # No duplicated qubit indices
        assert len(qubits) == len(set(qubits)), (
            f"{dataset_name}: Duplicate qubit index found."
        )

        terms_checked += 1

    print(f"  Canonical operators: {len(symbolic_terms)} ✓")

    datasets_checked += 1

print("\n" + "=" * 74)
print("PRE-STAGE 7F VALIDATION D PASSED")
print("=" * 74)

print(f"Datasets checked : {datasets_checked}")
print(f"Terms checked    : {terms_checked}")

print("\n✓ Qubits are stored in canonical order.")
print("✓ Pauli strings match canonical qubit ordering.")
print("✓ No duplicated qubit indices.")
print("✓ Duplicate detection is mathematically rigorous.")

In [ ]:
# ============================================================
# Stage 7F
# Runtime-Parameterized First-Order Operator Family
#
# Construct the reusable symbolic operator family for O1.
#
# The operator basis is fixed for a dataset.
#
# Runtime execution requires only the molecular parameter
# vector θ.
#
# The interpolation is fixed by construction and is therefore
# not represented as a runtime parameter.
# ============================================================

runtime_O1_operator_family = {}

for dataset_name, hamiltonian in O1_hamiltonians.items():

    runtime_terms = []

    for term in hamiltonian["pauli_terms"]:

        runtime_term = {

            # Fixed symbolic operator
            "pauli": term["pauli"],
            "qubits": term["qubits"],
            "locality": term["locality"],

            # Fixed commutator algebra
            "commutator_coefficient":
                term["commutator_coefficient"],

            "term_type":
                term["term_type"],
        }

        # --------------------------------------------------
        # Fixed interaction contribution
        # --------------------------------------------------

        if term["term_type"] == "interaction":

            runtime_term["interaction_index"] = (
                term["interaction_index"]
            )

            runtime_term["coefficient"] = (
                term["coefficient"]
            )

        # --------------------------------------------------
        # Runtime molecular contribution
        # --------------------------------------------------

        else:

            runtime_term["parameter_index"] = (
                term["parameter_index"]
            )

        runtime_terms.append(runtime_term)

    runtime_O1_operator_family[dataset_name] = {

        "num_qubits":
            hamiltonian["num_qubits"],

        "runtime_parameters": {

            "name": "theta",

            "parameterized": True,

            "interpolation_runtime_parameter": False,

        },

        "fixed_interpolation": {

            "type": "linear",

            "runtime_parameter": False,

        },

        "operator_family_fixed": True,

        "terms": runtime_terms,

    }

    interaction_terms = sum(
        t["term_type"] == "interaction"
        for t in runtime_terms
    )

    local_terms = sum(
        t["term_type"] == "local"
        for t in runtime_terms
    )

    print(dataset_name)
    print(f"Interaction operators : {interaction_terms}")
    print(f"Parameterized operators: {local_terms}")
    print(f"Total operators       : {len(runtime_terms)}")
    print("✓ Runtime-ready O1 operator family")
    print("✓ Runtime input: θ only")
    print("✓ λ fixed by construction")
    print("-" * 60)

In [ ]:
# ============================================================
# Stage 7G
# BUILD FULL ADIABATIC HAMILTONIAN
#
# H_ad(λ) = (1-λ) H_initial + λ H_problem
#
# Symbolically we retain both Hamiltonian components.
#
# Runtime evaluation supplies λ.
# ============================================================

from copy import deepcopy

Had_hamiltonians = {}

for dataset_name, H_problem in problem_operator_family.items():

    pauli_terms = []

    # --------------------------------------------------
    # Initial transverse-field Hamiltonian
    # H_initial = -Σ X_i
    # --------------------------------------------------

    num_qubits = H_problem["num_qubits"]

    for q in range(num_qubits):

        pauli_terms.append({

            "term_type": "initial",

            "coefficient": -1.0,

            "pauli": "X",

            "qubits": (q,),

            "locality": 1

        })

    # --------------------------------------------------
    # Problem Hamiltonian
    # --------------------------------------------------

    for term in H_problem["interaction_basis"]:

        new_term = deepcopy(term)

        new_term["term_type"] = "interaction"

        pauli_terms.append(new_term)

    for term in H_problem["local_basis"]:

        new_term = deepcopy(term)

        new_term["term_type"] = "local"

        pauli_terms.append(new_term)

    Had_hamiltonians[dataset_name] = {

        "hamiltonian_type": "H_ad",

        "formula":
            "H_ad(lambda) = (1-lambda) H_initial + lambda H_problem",

        "num_qubits":
            num_qubits,

        "runtime_parameter":
            "lambda",

        "pauli_terms":
            pauli_terms

    }

    print(
        dataset_name,
        "->",
        len(pauli_terms),
        "symbolic H_ad terms"
    )

In [ ]:
# ============================================================
# Stage 7H
# BUILD SECOND-ORDER OPERATOR
#
# O2 = [H_ad, O1]
#
# Symbolic commutator representation.
#
# Coefficients are not numerically combined here because
# H_ad and O1 retain runtime dependence on λ and θ.
# ============================================================

import pennylane as qml
from pennylane.pauli import PauliWord


def term_to_pauli_word(term):
    """
    Convert the notebook Pauli-term dictionary format into
    a PennyLane PauliWord.
    """

    return PauliWord({
        qubit: pauli
        for qubit, pauli in zip(
            term["qubits"],
            term["pauli"]
        )
    })


O2 = {}

for dataset_name, Had in Had_hamiltonians.items():

    print(f"Processing {dataset_name}...")

    if dataset_name not in O1_hamiltonians:
        raise KeyError(
            f"{dataset_name} is missing from O1_hamiltonians."
        )

    Had_terms = Had["pauli_terms"]
    O1_terms = O1_hamiltonians[dataset_name]["pauli_terms"]

    dataset_results = []

    for had_term in Had_terms:

        had_word = term_to_pauli_word(had_term)

        for o1_term in O1_terms:

            o1_word = term_to_pauli_word(o1_term)

            comm = qml.commutator(
                had_word,
                o1_word,
                pauli=True
            )

            if len(comm) == 0:
                continue

            dataset_results.append({

                # Exact Pauli commutator
                "commutator": comm,

                # Preserve complete source terms so Stage 7I
                # can construct the correct symbolic coefficient
                "had_term": had_term.copy(),

                "o1_term": o1_term.copy(),

            })

    O2[dataset_name] = dataset_results

    print(f"H_ad terms             : {len(Had_terms)}")
    print(f"O1 terms               : {len(O1_terms)}")
    print(f"Non-zero commutators   : {len(dataset_results)}")
    print("-" * 60)

In [ ]:
# ============================================================
# Stage 7I
# BUILD O2 HAMILTONIANS
#
# Convert symbolic commutators into the standard Hamiltonian
# representation used throughout the notebook.
# ============================================================

from copy import deepcopy

O2_hamiltonians = {}

for dataset_name, commutators in O2.items():

    pauli_terms = []

    for entry in commutators:

        comm = entry["commutator"]

        had_term = deepcopy(entry["had_term"])
        o1_term = deepcopy(entry["o1_term"])

        # Preserve the Pauli commutator coefficient separately.
        # The complete O2 coefficient is assembled later from:
        #     (H_ad coefficient)
        #   × (O1 coefficient)
        #   × (commutator coefficient)

        for pw, coeff in comm.items():

            qubits = tuple(sorted(pw.keys()))
            pauli = "".join(pw[q] for q in qubits)

            pauli_terms.append({

                "commutator_coefficient": complex(coeff),

                "had_term": had_term,

                "o1_term": o1_term,

                "pauli": pauli,

                "qubits": qubits,

                "locality": len(qubits)

            })

    O2_hamiltonians[dataset_name] = {

        "hamiltonian_type": "O2",

        "formula": "O2 = [H_ad, O1]",

        "num_qubits": Had_hamiltonians[dataset_name]["num_qubits"],

        "pauli_terms": pauli_terms

    }

    print(
        f"{dataset_name}: "
        f"{len(pauli_terms)} O2 Pauli terms"
    )

In [ ]:
# ============================================================
# Stage 7J
# Merge Identical O2 Operators
#
# Group identical Pauli operators while preserving every
# symbolic coefficient contribution.
#
# The complete coefficient is intentionally NOT assembled
# here because it still depends on:
#
#     • H_ad coefficient
#     • O1 coefficient
#     • commutator coefficient
#
# These are combined later during AGP evaluation.
# ============================================================

from collections import defaultdict
from copy import deepcopy

merged_O2_hamiltonians = {}

for dataset_name, hamiltonian in O2_hamiltonians.items():

    grouped_terms = defaultdict(list)

    # --------------------------------------------------------
    # Group identical Pauli operators
    # --------------------------------------------------------

    for term in hamiltonian["pauli_terms"]:

        key = (
            term["pauli"],
            term["qubits"],
        )

        grouped_terms[key].append({

            "commutator_coefficient":
                term["commutator_coefficient"],

            "had_term":
                deepcopy(term["had_term"]),

            "o1_term":
                deepcopy(term["o1_term"]),

        })

    merged_terms = []

    # --------------------------------------------------------
    # Build merged symbolic operator family
    # --------------------------------------------------------

    for (pauli, qubits), contributions in grouped_terms.items():

        merged_terms.append({

            "pauli":
                pauli,

            "qubits":
                qubits,

            "locality":
                len(qubits),

            # Every symbolic contribution is preserved.
            "contributions":
                contributions,

            "num_contributions":
                len(contributions),

        })

    merged_O2_hamiltonians[dataset_name] = {

        "hamiltonian_type":
            "Merged O2",

        "formula":
            "O2 = [H_ad, O1]",

        "num_qubits":
            hamiltonian["num_qubits"],

        "pauli_terms":
            merged_terms,

    }

    print(
        f"{dataset_name}: "
        f"{len(hamiltonian['pauli_terms'])} -> "
        f"{len(merged_terms)} unique O2 operators"
    )

    duplicate_count = sum(
        term["num_contributions"] - 1
        for term in merged_terms
    )

    print(
        f"Duplicate symbolic contributions merged: "
        f"{duplicate_count}"
    )

    print("-" * 60)

# Sanity check 

In [ ]:
# ============================================================
# Stage 7J Validation
# Lossless Symbolic Operator Merging
#
# Verify that Stage 7J performs only a regrouping of symbolic
# contributions and preserves the complete O2 operator.
# ============================================================

from collections import Counter

print("=" * 74)
print("VALIDATING MERGED O2 HAMILTONIANS")
print("Stage 7I  --->  Stage 7J")
print("=" * 74)

datasets_checked = 0

total_original_terms = 0
total_merged_operators = 0
total_contributions = 0

for dataset_name in O2_hamiltonians:

    print(f"\n{dataset_name}")

    original_terms = O2_hamiltonians[dataset_name]["pauli_terms"]
    merged_terms = merged_O2_hamiltonians[dataset_name]["pauli_terms"]

    # --------------------------------------------------------
    # Validation 1
    # Every original symbolic contribution must appear exactly
    # once after merging.
    # --------------------------------------------------------

    contribution_count = sum(
        len(term["contributions"])
        for term in merged_terms
    )

    assert contribution_count == len(original_terms), (
        f"{dataset_name}: "
        f"Expected {len(original_terms)} symbolic contributions "
        f"but found {contribution_count}."
    )

    # --------------------------------------------------------
    # Validation 2
    # Every merged operator must be unique.
    # --------------------------------------------------------

    merged_keys = [

        (
            term["pauli"],
            term["qubits"]
        )

        for term in merged_terms

    ]

    assert len(merged_keys) == len(set(merged_keys)), (

        f"{dataset_name}: Duplicate merged operators remain."

    )

    # --------------------------------------------------------
    # Validation 3
    # Operator multiplicities must match exactly.
    # --------------------------------------------------------

    original_counter = Counter(

        (
            term["pauli"],
            term["qubits"]
        )

        for term in original_terms

    )

    merged_counter = {

        (
            term["pauli"],
            term["qubits"]
        ):

        len(term["contributions"])

        for term in merged_terms

    }

    assert dict(original_counter) == merged_counter, (

        f"{dataset_name}: Operator multiplicities changed."

    )

    # --------------------------------------------------------
    # Validation 4
    # Every symbolic contribution still contains complete
    # coefficient metadata.
    # --------------------------------------------------------

    for merged_term in merged_terms:

        assert merged_term["num_contributions"] == len(
            merged_term["contributions"]
        )

        for contribution in merged_term["contributions"]:

            assert "commutator_coefficient" in contribution

            assert "had_term" in contribution

            assert "o1_term" in contribution

    print(f"Original O2 terms      : {len(original_terms)}")
    print(f"Merged operators       : {len(merged_terms)}")
    print(f"Symbolic contributions : {contribution_count}")
    print("✓ Lossless merge verified")

    total_original_terms += len(original_terms)
    total_merged_operators += len(merged_terms)
    total_contributions += contribution_count

    datasets_checked += 1

print("\n" + "=" * 74)
print("GLOBAL VALIDATION PASSED")
print("=" * 74)

print(f"Datasets validated      : {datasets_checked}")
print(f"Original O2 terms       : {total_original_terms}")
print(f"Merged operators        : {total_merged_operators}")
print(f"Symbolic contributions  : {total_contributions}")

assert total_original_terms == total_contributions

print("\n✓ No symbolic contributions lost.")
print("✓ No symbolic contributions duplicated.")
print("✓ Every merged operator is unique.")
print("✓ Symbolic coefficient metadata preserved.")
print("✓ Stage 7J is mathematically lossless.")

Experiemtn

In [ ]:
# ============================================================
# Stage 7K
# Assemble Symbolic O2 Coefficients
#
# Build the symbolic coefficient structure for every merged
# O2 operator.
#
# Each contribution is represented as
#
#     coefficient =
#         (H_ad coefficient)
#       * (O1 coefficient or parameter index)
#       * (commutator coefficient)
#
# Local O1 terms retain their parameter_index so that
# molecular feature values can be injected at runtime.
#
# Interaction terms retain their interaction coefficient.
#
# No runtime substitution is performed in this stage.
# ============================================================

assembled_O2_hamiltonians = {}

for dataset_name, hamiltonian in merged_O2_hamiltonians.items():

    assembled_terms = []

    for term in hamiltonian["pauli_terms"]:

        symbolic_contributions = []

        for contribution in term["contributions"]:

            symbolic_contributions.append({

                "had_term": contribution["had_term"],

                "o1_term": contribution["o1_term"],

                "commutator_coefficient":
                    contribution["commutator_coefficient"]

            })

        assembled_terms.append({

            "pauli": term["pauli"],

            "qubits": term["qubits"],

            "locality": term["locality"],

            "num_contributions": term["num_contributions"],

            "symbolic_contributions": symbolic_contributions

        })

    assembled_O2_hamiltonians[dataset_name] = {

        "hamiltonian_type": "Symbolic O2",

        "formula": "O2 = [H_ad, O1]",

        "num_qubits": hamiltonian["num_qubits"],

        "pauli_terms": assembled_terms,

    }

    print(
        f"{dataset_name}: "
        f"{len(assembled_terms)} symbolic O2 operators"
    )

# Submission artifact export
This explicit handoff replaces reliance on interactive kernel state.


In [ ]:
# Export the complete Stage-6 handoff for the counterdiabatic notebook.
from pathlib import Path
import pickle

root_candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
SUBMISSION_ROOT = next((p for p in root_candidates if (p / "requirements.txt").is_file()), None)
if SUBMISSION_ROOT is None:
    raise FileNotFoundError("Run inside HORIZON_PROJECTS_FINAL_SUBMISSION or its pipeline folder")
ARTIFACT_DIR = SUBMISSION_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

hamiltonian_A_interaction_ising = {}
hamiltonian_B_diagonal_feature = {}
for name, HA in global_hamiltonian.items():
    hypergraph = interaction_hypergraph[name]
    selected = selected_feature_data[name]
    active = list(hypergraph["active_features"])
    original = [int(hypergraph["original_feature_map"][i]) for i in range(len(active))]
    interaction_terms = []
    for term in HA["terms"]:
        interaction_terms.append({
            "coefficient": float(term["coefficient"]),
            "pauli_string": " ".join(f"{p}{q}" for p, q in zip(term["pauli"], term["qubits"])),
            "locality": int(term["locality"]),
        })
    local_terms = []
    for q, position in enumerate(active):
        local_terms.append({
            "coefficient": float(selected["selected_mi_scores"][position]),
            "pauli_string": f"Z{q}",
            "locality": 1,
            "parameter_index": q,
        })
    hamiltonian_A_interaction_ising[name] = {
        "hamiltonian_type": "global_interaction",
        "formula": "H_A = sum J_ij ZiZj + sum K_ijk ZiZjZk",
        "selected_features": original,
        "pauli_terms": interaction_terms,
    }
    hamiltonian_B_diagonal_feature[name] = {
        "hamiltonian_type": "parameterized_local",
        "formula": "H_B(theta) = sum theta_i Zi",
        "selected_features": original,
        "selected_mi_scores": selected["selected_mi_scores"][active],
        "X_selected_scaled": selected["X_selected_scaled"][:, active],
        "pauli_terms": local_terms,
    }

curated_admet_hamiltonians = {
    "schema_version": 1,
    "hamiltonian_A_interaction_ising": hamiltonian_A_interaction_ising,
    "hamiltonian_B_diagonal_feature": hamiltonian_B_diagonal_feature,
}
artifact_path = ARTIFACT_DIR / "curated_admet_hamiltonians.pkl"
with artifact_path.open("wb") as handle:
    pickle.dump(curated_admet_hamiltonians, handle)
print("Exported:", artifact_path)
